# Flat-guide GFMC on the $6\times 6$ Kagome torus

Run this notebook from the repository root, or from `notebook/`. The solver is `code/gfmc/`. The graph and the disorder come from `code/kagome_qea/`. The projector is the same continuous-time Green's function Monte Carlo as `notebook/tfim_gfmc.ipynb`, with guiding function $\psi = 1$.

The operator is

$$
H = -\sum_b J_b \sigma_i^z \sigma_j^z - \Gamma \sum_i \sigma_i^x - h \sum_i \sigma_i^z.
$$

$s_i = \pm 1$. This run is the round-1 Gaussian nearest-neighbor ensemble: each stored bond has an independent $J_b \sim N(0,1)$, and $h = 0$. $\Gamma$ is a solver argument. It is not stored in the disorder.

`build_kagome(6, periodic=True)` is six by six three-site cells on the torus: $N = 3 \times 6^2 = 108$ sites, $6 \times 6^2 = 216$ bonds, and degree 4 at every site. A dense $2^{108}$ matrix is not built. Exact diagonalization in this repository stops at 16 sites.


## What this run is

One periodic sample, `sample_id = 0`. The disorder stream is PCG64 with entropy `[1000, 6, 0, 1]`. The projector uses its own seed, `0`. Open and periodic realizations are different streams. The open $6\times 6$ patch is counted below and is not projected.

The $\Gamma$ grid is the round-1 window, seven values from $0.5$ to $3.5$. Each value is one population of 2500 walkers. `n_walkers` is that population size, not the power in $(\Lambda - H)^n$. The projection length is $(40 + 160) \times 0.2 = 40$, and the reported energies use the last 160 blocks. A short timing on this machine gave about $0.27$ s per block at $\Gamma = 1$, and the hop rate is $N\Gamma$, so the grid is on the order of 10–15 minutes. Larger $\Gamma$ takes longer.

This is not a disorder average and not a finite-size crossing. The error is the autocorrelation-aware standard error from `series_error`. `resolved` means the measured series has at least 64 values, its positive sequence terminated, and it has at least 8 estimated independent samples. That flag is a length check, not a coverage certificate.

A finished row is left on disk. Running the cell again loads it.

```text
data/kagome/gaussian_nn/pbc/L6/gfmc/h0/gamma{gamma}/sample0_seed0.npz
```

A decimal point in `gamma` is written as `p`.


In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

root = Path.cwd()
if not (root / "code").exists():
    root = root.parent
sys.path.insert(0, str(root / "code"))

from gfmc import GFMC
from kagome_qea import build_kagome, make_sample

L = 6
SAMPLE_ID = 0
GAMMAS = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5)
FIELD = 0.0
PROJECTOR_SEED = 0
SETTINGS = dict(n_walkers=2500, tau_block=0.2, n_equilibrate=40, n_measure=160)

def token(value):
    text = f"{float(value):.8f}".rstrip("0").rstrip(".")
    if text in ("", "-0"):
        text = "0"
    return text.replace("-", "m").replace(".", "p")

def run_path(gamma):
    directory = (
        root / "data" / "kagome" / "gaussian_nn" / "pbc"
        / f"L{L}" / "gfmc" / f"h{token(FIELD)}" / f"gamma{token(gamma)}"
    )
    return directory / f"sample{SAMPLE_ID}_seed{PROJECTOR_SEED}.npz"


In [ ]:
def summarize(periodic):
    points, bonds, triangles, adjacency = build_kagome(L, periodic=periodic)
    degrees = np.asarray(adjacency.sum(axis=1), dtype=int)
    counts = {int(degree): int(np.count_nonzero(degrees == degree)) for degree in np.unique(degrees)}
    print(
        f"{'pbc' if periodic else 'obc':<4} sites {points.shape[0]:3d}  "
        f"bonds {bonds.shape[0]:3d}  triangles {len(triangles):3d}  degrees {counts}"
    )
    return points, bonds, triangles, degrees

open_points, open_bonds, open_triangles, open_degrees = summarize(False)
points, bonds, triangles, degrees = summarize(True)
if open_points.shape[0] != 108 or open_bonds.shape[0] != 193 or len(open_triangles) != 61:
    raise RuntimeError("open 6x6 counts are not 108 sites, 193 bonds, 61 triangles")
if points.shape[0] != 108 or bonds.shape[0] != 216 or len(triangles) != 72 or not np.all(degrees == 4):
    raise RuntimeError("periodic 6x6 counts are not 108 sites, 216 bonds, degree 4")

sample = make_sample(L, True, SAMPLE_ID)
print(
    f"sample {sample.sample_id}  entropy {sample.seed_entropy.tolist()}  "
    f"J mean {sample.J.mean():.4f}  J std {sample.J.std():.4f}"
)
operator = sample.hamiltonian(FIELD)


## Energies

Each row is the ground-state energy of this one sample. The mixed number is the weight-and-time average of the local energy. The growth number is $E_T - \log(W / n_{\mathrm{walkers}}) / \tau_{\mathrm{block}}$. Both estimate the same ground-state energy for $\psi = 1$. The table also prints the energy per site.


In [ ]:
def load_row(path):
    with np.load(path, allow_pickle=False) as data:
        row = {}
        for key in data.files:
            value = data[key]
            row[key] = value.item() if value.shape == () else np.array(value)
        return row

rows = []
for gamma in GAMMAS:
    path = run_path(gamma)
    if path.exists():
        row = load_row(path)
        print(f"loaded  Γ={gamma:.1f}  {path.relative_to(root)}", flush=True)
    else:
        started = time.perf_counter()
        result = GFMC(operator, gamma, h=FIELD).run(seed=PROJECTOR_SEED, **SETTINGS)
        seconds = time.perf_counter() - started
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            path,
            L=np.int64(sample.L),
            periodic=np.bool_(sample.periodic),
            sample_id=np.int64(sample.sample_id),
            ensemble=np.array(sample.ensemble),
            seed_entropy=np.asarray(sample.seed_entropy, dtype=np.int64),
            gamma=np.float64(gamma),
            h=np.float64(FIELD),
            seed=np.int64(PROJECTOR_SEED),
            n_walkers=np.int64(SETTINGS["n_walkers"]),
            tau_block=np.float64(SETTINGS["tau_block"]),
            n_equilibrate=np.int64(SETTINGS["n_equilibrate"]),
            n_measure=np.int64(SETTINGS["n_measure"]),
            n_sites=np.int64(sample.n_sites),
            seconds=np.float64(seconds),
            energy=np.float64(result["energy"]),
            energy_error=np.float64(result["energy_error"]),
            energy_growth=np.float64(result["energy_growth"]),
            energy_growth_error=np.float64(result["energy_growth_error"]),
            energy_resolved=np.bool_(result["energy_resolved"]),
            energy_growth_resolved=np.bool_(result["energy_growth_resolved"]),
            energy_series=np.asarray(result["energy_series"], dtype=np.float64),
            energy_growth_series=np.asarray(result["energy_growth_series"], dtype=np.float64),
            J=np.asarray(sample.J, dtype=np.float64),
        )
        row = load_row(path)
        print(
            f"ran     Γ={gamma:.1f}  {seconds:6.1f}s  "
            f"mixed {result['energy']:.4f} ± {result['energy_error']:.4f}  "
            f"resolved {result['energy_resolved']}",
            flush=True,
        )
    rows.append(row)

n_sites = int(rows[0]["n_sites"])
header = (
    f"{'Γ':>5} {'mixed':>10} {'err':>8} {'growth':>10} {'err':>8} "
    f"{'E/N':>8} {'flag':>5} {'growth flag':>11}"
)
print(header)
for row in rows:
    print(
        f"{row['gamma']:5.1f} {row['energy']:10.4f} {row['energy_error']:8.4f} "
        f"{row['energy_growth']:10.4f} {row['energy_growth_error']:8.4f} "
        f"{row['energy'] / n_sites:8.4f} {str(bool(row['energy_resolved'])):>5} "
        f"{str(bool(row['energy_growth_resolved'])):>11}"
    )


In [ ]:
import matplotlib.pyplot as plt

gammas = np.array([row["gamma"] for row in rows], dtype=float)
mixed = np.array([row["energy"] for row in rows], dtype=float) / n_sites
mixed_error = np.array([row["energy_error"] for row in rows], dtype=float) / n_sites
growth = np.array([row["energy_growth"] for row in rows], dtype=float) / n_sites
growth_error = np.array([row["energy_growth_error"] for row in rows], dtype=float) / n_sites

figure, axis = plt.subplots(figsize=(6.0, 4.0))
axis.errorbar(gammas, mixed, yerr=mixed_error, fmt="o", capsize=3, label="mixed")
axis.errorbar(gammas + 0.03, growth, yerr=growth_error, fmt="s", capsize=3, label="growth")
axis.set_xlabel(r"$\Gamma$")
axis.set_ylabel("$E / N$")
axis.set_title(r"$6\times 6$ Kagome torus, one Gaussian sample, $h = 0$")
axis.legend()
figure.tight_layout()
plt.show()
